# Notebook Title: Exploratory Data Analysis (EDA) - Student & Career Profiles

## Purpose
"Exploratory Data Analysis of raw and refined student profiles" 
To validate data integrity, identify underlying distributions across the 11-point vector space, and confirm the statistical readiness of the datasets for Machine Learning model training.

## Context
- **Project Phase:** Exploration / Research
- **Related Module(s):** `src/data_processing`, `src/research`

## Data Sources
- **External**
- **Raw Kaggle Dataset:** Student personality survey responses (TIPI and RIASEC).
- **Refined Vector Library:** Normalized 11-point vector profiles (6 RIASEC + 5 Big Five) derived from the raw Kaggle data and O*NET Career Work Styles.

## Assumptions & Constraints
- **Assumptions:** Survey responses are considered accurate reflections of user traits; O*NET mappings to Big Five are psychologically sound.
- **Constraints:** Analysis is limited to the 11 dimensions defined in the unified vector space; original Kaggle data may contain noise or missing values that require filtering.

## Reproducibility
- **Environment:** Local (VS Code / Jupyter / Anaconda)

## Expected Outputs
- **Distribution Plots:** Histograms showing the spread of RIASEC and Big Five scores.
- **Correlation Heatmap:** Visualizing the relationship between interests and personality traits.
- **PCA Visualization:** 2D mapping of the 11D vector space to identify clusters and data coverage.
- **Data Health Metrics:** Missing value counts and normalization range checks (0.0 to 1.0).

## Notes
- This notebook acts as a "Data Health Check" to ensure that the normalization and feature alignment between the Kaggle (Student) and O*NET (Career) datasets are mathematically consistent before training the Random Forest model.

In [17]:
import pandas as pd

# Load the RAW Kaggle data
raw_data_path = Path.cwd() / "docs/data/raw_kaggle_data.csv"
if not raw_data_path.exists():
    raw_data_path = Path.cwd().parent / "docs/data/raw_kaggle_data.csv"

# Try reading with a tab separator
raw_df = pd.read_csv(raw_data_path, sep='\t')

print("="*50)
print("RAW KAGGLE DATASET: HEALTH CHECK")
print("="*50)

print("\n1. Original Shape:")
print(raw_df.shape)

print("\n2. Missing Values in Raw Data:")
raw_missing = raw_df.isnull().sum()
print(raw_missing[raw_missing > 0]) # Only show columns with nulls

print("\n3. Original Value Ranges (Notice it's not 0.0 to 1.0):")
# Pick a few columns to look at the original min/max
raw_df

RAW KAGGLE DATASET: HEALTH CHECK

1. Original Shape:
(145828, 94)

2. Missing Values in Raw Data:
country            12
major           52874
Unnamed: 93    145827
dtype: int64

3. Original Value Ranges (Notice it's not 0.0 to 1.0):


C:\Users\grosh\AppData\Local\Temp\ipykernel_77416\2880166162.py:9: DtypeWarning: Columns (93) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(raw_data_path, sep='\t')


In [16]:
raw_df.columns

Index(['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'I1', 'I2', 'I3', 'I4',
       'I5', 'I6', 'I7', 'I8', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8',
       'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'E1', 'E2', 'E3', 'E4',
       'E5', 'E6', 'E7', 'E8', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8',
       'introelapse', 'testelapse', 'surveyelapse', 'TIPI1', 'TIPI2', 'TIPI3',
       'TIPI4', 'TIPI5', 'TIPI6', 'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'VCL1',
       'VCL2', 'VCL3', 'VCL4', 'VCL5', 'VCL6', 'VCL7', 'VCL8', 'VCL9', 'VCL10',
       'VCL11', 'VCL12', 'VCL13', 'VCL14', 'VCL15', 'VCL16', 'education',
       'urban', 'gender', 'engnat', 'age', 'hand', 'religion', 'orientation',
       'race', 'voted', 'married', 'familysize', 'uniqueNetworkLocation',
       'country', 'source', 'major', 'Unnamed: 93'],
      dtype='object')

In [24]:
refined_kaggle_df = pd.read_csv(data_path)
refined_kaggle_df.columns


Index(['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8', 'I1', 'I2', 'I3', 'I4',
       'I5', 'I6', 'I7', 'I8', 'A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8',
       'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'E1', 'E2', 'E3', 'E4',
       'E5', 'E6', 'E7', 'E8', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8',
       'Realistic', 'Investigative', 'Artistic', 'Social', 'Enterprising',
       'Conventional', 'TIPI1', 'TIPI2', 'TIPI3', 'TIPI4', 'TIPI5', 'TIPI6',
       'TIPI7', 'TIPI8', 'TIPI9', 'TIPI10', 'Extraversion', 'Agreeableness',
       'Conscientiousness', 'Emotional_Stability', 'Openness', 'major'],
      dtype='object')

In [25]:
refined_kaggle_df.isnull().sum()

R1                         0
R2                         0
R3                         0
R4                         0
R5                         0
                       ...  
Agreeableness              0
Conscientiousness          0
Emotional_Stability        0
Openness                   0
major                  52874
Length: 70, dtype: int64

In [26]:
refined_kaggle_df.describe()

,R1,R2,R3,R4,R5,R6,R7,R8,I1,I2,...,TIPI6,TIPI7,TIPI8,TIPI9,TIPI10,Extraversion,Agreeableness,Conscientiousness,Emotional_Stability,Openness
count,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,...,145828.000000,145828.000000,145828.00000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000,145828.000000
mean,0.393212,0.276715,0.187185,0.323410,0.187486,0.299454,0.254255,0.241804,0.608688,0.583917,...,4.467592,5.592815,3.06784,4.997092,2.862009,0.517337,0.621150,0.676249,0.567457,0.705387
std,0.329587,0.308803,0.280218,0.334789,0.268044,0.318387,0.295710,0.296127,0.333297,0.340717,...,1.950515,1.454636,1.85829,1.646894,1.734016,0.233229,0.178987,0.200023,0.216672,0.169432
min,-0.250000,-0.250000,-0.250000,-0.250000,-0.250000,-0.250000,-0.250000,-0.250000,-0.250000,-0.250000,...,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.500000,0.250000,...,3.000000,5.000000,1.00000,4.000000,1.000000,0.357143,0.500000,0.500000,0.428571,0.571429
50%,0.500000,0.250000,0.000000,0.250000,0.000000,0.250000,0.250000,0.250000,0.750000,0.750000,...,5.000000,6.000000,3.00000,5.000000,2.000000,0.500000,0.642857,0.714286,0.571429,0.714286
75%,0.500000,0.500000,0.250000,0.500000,0.250000,0.500000,0.500000,0.500000,1.000000,0.750000,...,6.000000,7.000000,5.00000,6.000000,4.000000,0.714286,0.785714,0.857143,0.714286,0.857143
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,7.000000,7.000000,7.00000,7.000000,7.000000,1.000000,1.000000,1.000000,1.000000,1.000000
